# TRAKE — Eval Tier3 + Qwen (Run 4/5/6) tren Colab GPU

Chay 3 tang + Qwen2-VL tren **Colab GPU**, khong can Mongo/UI. Can: **Qdrant collection `jina`** (ban zip tu `collections/jina`, Qdrant **1.19.0**, 921k diem) + **video goc** cua video_id lam GT.

**Truoc khi chay:** Runtime -> Change runtime type -> **T4 GPU**.


## 1. Kiem tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Chua bat GPU! Runtime -> Change runtime type -> T4 GPU"


## 2. Lay code (nhanh feature/overral-trake-huy)

In [ ]:
%cd /content
![ -d aic2026 ] || git clone https://github.com/khoinguyen248/aic2026.git
%cd /content/aic2026
!git fetch -q origin && git checkout feature/overral-trake-huy && git pull -q


## 3. Cai phu thuoc (GIU torch/numpy/scipy co san cua Colab)

KHONG pin numpy/scipy/Pillow/torch (de tranh conflict voi stack Colab). Chi pin cai that su can.

In [ ]:
!pip -q install flask==3.0.3 flask-cors flask-pymongo python-dotenv==1.0.1 \
  einops==0.8.1 timm==0.4.12 opencv-python-headless ftfy sentencepiece \
  transformers==4.55.1 open-clip-torch torchscale \
  qdrant-client pymongo deep-translator accelerate bitsandbytes qwen-vl-utils gdown
print("deps OK")


## 4. Qdrant 1.19.0 + restore collection `jina`

**Sua `JINA_ZIP_URL`** = link zip `jina` cua ban (host de wget, hoac Google Drive o duoi).
Zip phai chua `config.json`, `shard_key_mapping.json`, `version.info`, thu muc `0/` (noi dung cua `collections/jina`).

In [ ]:
import os, subprocess, glob, shutil, time, re, zipfile
%cd /content
# 4a. binary Qdrant 1.19.0 (khop version tao data)
!wget -q https://github.com/qdrant/qdrant/releases/download/v1.19.0/qdrant-x86_64-unknown-linux-musl.tar.gz -O qdrant.tgz
!tar xzf qdrant.tgz
assert os.path.isfile("/content/qdrant"), "Khong tai duoc binary qdrant 1.19.0"

# 4b. lay zip jina -- HO TRO Google Drive (link share) + http truc tiep + file local
JINA_ZIP_URL = "https://drive.google.com/file/d/1QCnx_Ksbr06X-HDCrFn95d2nku_MR2MD/view?usp=sharing"  # <-- SUA neu can
JINA_ZIP = "/content/jina.zip"
if "drive.google.com" in JINA_ZIP_URL:
    m = re.search(r"/d/([A-Za-z0-9_-]+)", JINA_ZIP_URL) or re.search(r"id=([A-Za-z0-9_-]+)", JINA_ZIP_URL)
    assert m, "Khong tach duoc file id tu link Drive"
    import gdown
    gdown.download(id=m.group(1), output=JINA_ZIP, quiet=False)   # tu xu ly file lon + confirm token
elif JINA_ZIP_URL.startswith("http"):
    !wget -q "$JINA_ZIP_URL" -O "$JINA_ZIP"
else:
    JINA_ZIP = JINA_ZIP_URL  # duong dan local (vd sau khi mount Drive)

# kiem tra la file ZIP that (tranh truong hop tai nham trang HTML)
assert os.path.isfile(JINA_ZIP), f"Khong co file {JINA_ZIP}"
sz = os.path.getsize(JINA_ZIP)
assert zipfile.is_zipfile(JINA_ZIP), (
    f"File tai ve KHONG phai zip (size={sz}). Neu dung Drive: dat quyen 'Anyone with the link', "
    f"hoac dung o mount Drive o duoi.")
print("zip OK, size =", round(sz/1e6, 1), "MB")

# 4c. giai nen & dat dung cho: /content/qdrant_storage/collections/jina/
shutil.rmtree("/content/_jina_tmp", ignore_errors=True); os.makedirs("/content/_jina_tmp", exist_ok=True)
!unzip -q "$JINA_ZIP" -d /content/_jina_tmp
cfg = glob.glob("/content/_jina_tmp/**/config.json", recursive=True)
assert cfg, "Khong thay config.json trong zip -- kiem tra ban zip dung thu muc collections/jina"
src = os.path.dirname(cfg[0])
dst = "/content/qdrant_storage/collections/jina"
shutil.rmtree("/content/qdrant_storage", ignore_errors=True); os.makedirs(dst, exist_ok=True)
for name in os.listdir(src):
    shutil.move(os.path.join(src, name), os.path.join(dst, name))
print("collection files:", os.listdir(dst))


In [ ]:
# 4d. chay Qdrant nen
env = dict(os.environ); env["QDRANT__STORAGE__STORAGE_PATH"] = "/content/qdrant_storage"
qproc = subprocess.Popen(["/content/qdrant"], env=env,
                         stdout=open("/content/qdrant.log", "w"), stderr=subprocess.STDOUT)
time.sleep(12)
from qdrant_client import QdrantClient
c = QdrantClient(url="http://localhost:6333", check_compatibility=False)
print("collections:", [x.name for x in c.get_collections().collections])
print("jina count:", c.count("jina"))


> **(Tuy chon) Google Drive** neu zip tren Drive:
> ```python
> from google.colab import drive; drive.mount('/content/drive')
> # roi dat JINA_ZIP_URL = "/content/drive/MyDrive/jina.zip" o Cell 4b
> ```

## 5. Tai video goc (L25 / L26 / M02)

Mac dinh tai **L25**. O nay liet ke cac mp4 de biet video_id.

In [ ]:
%cd /content
VIDEO_ZIPS = [
    "https://aic-data.ledo.io.vn/Videos_L26_a.zip",
    # "https://aic-data.ledo.io.vn/Videos_L27_a.zip",   # <-- them link L27 cua ban
]
import glob
os.makedirs("/content/videos_raw", exist_ok=True)
for u in VIDEO_ZIPS:
    fn = "/content/" + u.split("/")[-1]
    !wget -q "$u" -O "$fn"
    !unzip -q -o "$fn" -d /content/videos_raw
mp4s = glob.glob("/content/videos_raw/**/*.mp4", recursive=True)
print(len(mp4s), "mp4:")
for p in sorted(mp4s)[:60]: print(" ", p)


## 6. Dua cac video_id co GT vao VIDEO_ROOT ten `<video_id>.mp4`

In [ ]:
import glob, shutil, os
VIDEO_IDS = ["L26_V069", "L27_V016"]   # video_id co GT & co mp4 trong videos_raw
os.makedirs("/content/videos", exist_ok=True)
allmp4 = glob.glob("/content/videos_raw/**/*.mp4", recursive=True)
for vid in VIDEO_IDS:
    key = vid.replace("_", "").replace("-", "").lower()
    cand = [p for p in allmp4 if key in os.path.basename(p).replace("_", "").replace("-", "").lower()]
    if not cand:
        print(f"  ! CHUA thay mp4 cho {vid} -> Tier3 bo qua video nay (Tier1/2 van chay)")
        continue
    shutil.copy(cand[0], f"/content/videos/{vid}.mp4")
    print(f"  OK {vid} <- {cand[0]}")
print("videos:", os.listdir("/content/videos"))


## 7. GT tu 2 cau co dap an (L26_V069 nau luon, L27_V016 thien nhien)

q3 co OCR cue "Canh ga lac" o E3 de kich hoat Run3 multimodal.

In [ ]:
import json, os
gt = [
  {
    "id": "q_L26_V069_nau_luon",
    "events": [
      "người đầu bếp lấy cọ phết dầu ăn lên chảo",
      "người đầu bếp đặt lươn lên chảo nướng",
      "người đầu bếp dùng muôi gỗ ép miếng lươn",
      "miếng lươn được đặt lên dĩa"
    ],
    "language": False,
    "model": "jina",
    "max_event_gap_s": None,
    "gt": {"video_id": "L26_V069", "frame_ids": [5360, 5473, 5631, 6104], "fps": 25}
  },
  {
    "id": "q_L27_V016_thien_nhien",
    "events": [
      "một cậu bé cưỡi trâu ở dưới nước và vung nước lên",
      "một người đàn ông đeo đèn pin trên đầu có quai xanh nước biển, đeo đồng hồ đen và cười, phía sau là cánh đồng lúa",
      "một cô gái mặc áo ngoài màu trắng đi giữa khu chợ ẩm thực có xe bán cánh gà lắc",
      "một chiếc vỏ lãi màu xanh da trời chạy trên con kênh, trên bờ có con đường và rừng"
    ],
    "events_ocr": ["", "", "Cánh gà lắc", ""],
    "language": False,
    "model": "jina",
    "max_event_gap_s": None,
    "gt": {"video_id": "L27_V016", "frame_ids": [86, 146, 283, 1197], "fps": 25}
  }
]
os.makedirs("/content/aic2026/backendAIC2025/app/eval/out", exist_ok=True)
p = "/content/aic2026/backendAIC2025/app/eval/queries_gt.json"
json.dump(gt, open(p, "w"), ensure_ascii=False, indent=2)
print("saved", p, "-", len(gt), "queries")


## 8. Chay harness -- Run 2 / 4 / 5 / 6 (Qwen bat, GPU)

In [ ]:
%cd /content/aic2026/backendAIC2025
%env USE_TF=0
%env TRANSFORMERS_NO_TF=1
%env QDRANT_URL=http://localhost:6333
%env SEARCH_DEVICE=cuda
%env VIDEO_ROOT=/content/videos
%env KEYFRAMES_PATH=/content/keyframes
%env TRAKE_TIER3_ENABLED=true
%env TRAKE_QWEN_RERANK_ENABLED=true
%env QWEN_MODEL_PATH=Qwen/Qwen2-VL-2B-Instruct
%env QWEN_QUANTIZATION=4bit
%env QWEN_DEVICE_MAP=cuda
%env TRAKE_TOP_M=300
!python -m app.eval.eval_trake --queries app/eval/queries_gt.json \
   --out app/eval/out --runs run2_dp,run4_tier3_argmax,run5_tier3_peak,run6_full \
   --k 1,5,10,100 --tol-frames 25


In [ ]:
print(open("/content/aic2026/backendAIC2025/app/eval/out/eval_results.md", encoding="utf-8").read())


## Ghi chu
- Phai thay `tier3_globally_ready=True` va `qwen_globally_ready=True`. Neu False: kiem tra video `/content/videos/<video_id>.mp4` va GPU.
- **Latency o day la so cua Colab (T4).** Run1/Run3 (greedy, multimodal can Atlas) chay tren may Docker local; ghi ro hardware tung bang, dung Run2 lam moc quy doi.
- `VideoHit=0` (Tang 1 khong lay dung video): tang `TRAKE_TOP_M` (Cell 8) hoac viet mo ta su kien ro hon.
- Kendall/Spearman ~ 1 do DP ep don dieu.
